# Session 7 — Bellman, DP, Monte Carlo, TD

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Concept · Explain → practice together

## How to use this notebook

1. Read **Part I** (topic) before coding.  
2. Every stop: **explain markdown → small code cell → run → talk**.  

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Code is ready. Run. Do not edit. |
| **TRY THIS** | You type `<<< FILL`. Re-run. Check. |
| **CHANGE ME** | Change the marked value. Re-run. **PAIR 20s** discuss. |
| **CHECK** | Print PASS / NOT YET |

![Bellman backup](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/bellman.png)
![Inventory MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

*All diagrams load from GitHub raw URLs (work in Colab and locally).*


# Part I — Topic (explain before coding)

## Same Electronics desk, new decide box

Sessions 5–6 defined the inventory **MDP** and the go/no-go memo.  
Today the decide box is a **value** — not a tool call.

```text
observe inventory → decide (value / best order) → demand → reward → next inventory
```

| Idea | Meaning |
| --- | --- |
| **V(s)** | Expected discounted reward from stock `s` if we keep acting this way |
| **Bellman backup** | Immediate reward + γ × next state's value, averaged with **known** P(demand) |
| **Value iteration** | Repeat the backup until the table settles |
| **Monte Carlo** | Wait for the episode end; assign the return |
| **TD** | Update after one step using the next state's current value |

### Model-based vs model-free (Lab 4 bridge)

| Approach | Needs P(demand)? | Tonight |
| --- | --- | --- |
| Value / Policy Iteration | **Yes** — full model | Session 7 (whiteboard + small desk) |
| **Q-learning** | **No** — learns from samples | Session 8 |

**Teach aloud:** A value is not this morning's profit. DP is clean when the model is known. On a real warehouse, Session 8 trades the table for **samples**.

**JUMP BACK** — do not open Stop 0 until I say OPEN.


---
# Part II — Practice (Electronics inventory)

Real log: `inventory_dataset_full.csv` → **Electronics** days (same file as Sessions 5–6).


## Stop 0 — Imports + settings (RUN WITH ME)


In [ ]:
# STOP 0 — RUN WITH ME · imports

from pathlib import Path
from collections import Counter
import csv
import statistics

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Electronics"


In [ ]:
# STOP 1a — RUN WITH ME · course_data

def course_data(name: str) -> Path:
    # Prefer Drive, then local folders, then GitHub download (Colab-safe)
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    if name in {"inventory_dataset_full.csv", "inventory_dataset.csv"}:
        import urllib.request
        dest = Path("/content/data") if Path("/content").exists() else Path("data")
        dest.mkdir(parents=True, exist_ok=True)
        out = dest / name
        url = "https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
        print("Downloading", name, "from GitHub…")
        urllib.request.urlretrieve(url, out)
        return out
    raise FileNotFoundError("Could not find " + name)


## Stop 1 — Load Electronics and build empirical P(d)


In [ ]:
# STOP 1a — RUN WITH ME · load Electronics rows

def _flag(val) -> bool:
    return str(val).strip() in {"1", "True", "true", "YES", "Yes"}


def load_electronics_desk():
    rows = []
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product") or "").strip() != PRODUCT:
                continue
            rows.append({
                "Date": row["Date"],
                "Current_Inventory": int(float(row["Current_Inventory"])),
                "Demand": int(float(row["Demand"])),
                "Stockout_Units": int(float(row["Stockout_Units"])),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
                "Profit": float(row.get("Profit") or 0),
                "Holding_Cost": float(row.get("Holding_Cost") or 0),
                "Stockout_Cost": float(row.get("Stockout_Cost") or 0),
                "Is_Promotional": _flag(row.get("Is_Promotional", 0)),
            })
    if not rows:
        raise ValueError("No Electronics rows found")
    rows.sort(key=lambda r: r["Date"])
    return rows


ROWS = load_electronics_desk()
N_DAYS = len(ROWS)
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
print("Electronics days:", N_DAYS, "| avg demand:", round(AVG_DEMAND, 1))


In [ ]:
# STOP 1b — RUN WITH ME · demand bins → P(d) for Bellman

demands = sorted(r["Demand"] for r in ROWS)
q1, q2 = demands[len(demands) // 3], demands[(2 * len(demands)) // 3]


def demand_bin(d: int) -> int:
    if d <= q1:
        return 0
    if d <= q2:
        return 1
    return 2


_counts = Counter(demand_bin(r["Demand"]) for r in ROWS)
DEMAND_P = {k: _counts[k] / N_DAYS for k in (0, 1, 2)}
print("P(d) from Electronics:", {k: round(v, 3) for k, v in DEMAND_P.items()})

if plt is not None:
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.bar(["0 low", "1 mid", "2 high"], [DEMAND_P[0], DEMAND_P[1], DEMAND_P[2]], color="#2F5D50")
    ax.set_ylabel("P(d)")
    ax.set_title("Empirical demand bins (Electronics)")
    ax.set_ylim(0, 1)
    fig.tight_layout()
    plt.show()


## Stop 2 — Bellman backup + value iteration (model known)

![Bellman](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/bellman.png)

Tiny MDP for the board: stock in 0..CAPACITY, order in {0,1,2}, demand bin in {0,1,2} with `DEMAND_P`.


In [ ]:
# STOP 2a — RUN WITH ME · transition + Bellman backup

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5


def transition(stock, order, demand):
    # demand here is the BIN label 0/1/2 mapped to a representative quantity
    demand_qty = [max(1, int(q1)), int((q1 + q2) / 2), int(q2 + (demands[-1] - q2) / 2) + 1][demand]
    # scale to fit tiny CAPACITY world (board-sized MDP)
    demand_qty = min(CAPACITY, max(0, demand_qty * CAPACITY // max(1, int(AVG_DEMAND))))
    started = min(CAPACITY, stock + order)
    sales = min(started, demand_qty)
    ending = started - sales
    missed = max(0, demand_qty - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay


def bellman_backup(values, stock, order):
    # Expected reward + gamma * V(next), averaged with known P(d)
    total = 0.0
    for demand, prob in DEMAND_P.items():
        ending, pay = transition(stock, order, demand)
        total += prob * (pay + GAMMA * values[ending])
    return total


print("Model known. CAPACITY =", CAPACITY, "GAMMA =", GAMMA)


In [ ]:
# STOP 2b — RUN WITH ME · value iteration

values = np.zeros(CAPACITY + 1)
for sweep in range(40):
    new_values = np.zeros_like(values)
    for stock in range(CAPACITY + 1):
        new_values[stock] = max(bellman_backup(values, stock, order) for order in range(3))
    values = new_values

policy = []
print("Value of each stock under best order (model known):")
for stock in range(CAPACITY + 1):
    scores = [bellman_backup(values, stock, order) for order in range(3)]
    best = int(np.argmax(scores))
    policy.append(best)
    print(f"  stock {stock}: V={values[stock]:.2f}  best order={best}")

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.0))
    axes[0].bar(range(CAPACITY + 1), values, color="#2F5D50")
    axes[0].set_xlabel("stock s")
    axes[0].set_ylabel("V(s)")
    axes[0].set_title("Value iteration")
    axes[1].bar(range(CAPACITY + 1), policy, color="#1F4E79")
    axes[1].set_xlabel("stock s")
    axes[1].set_ylabel("best order a")
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_title("Greedy policy from V")
    fig.tight_layout()
    plt.show()


### 2c — TRY THIS · empty shelf order

**PAIR 20s.** Look at the table. What order does value iteration place when stock = 0?


In [ ]:
# STOP 2c — TRY THIS

order_at_empty = None  # <<< FILL: 0, 1, or 2
learned = int(np.argmax([bellman_backup(values, 0, order) for order in range(3)]))
print("You said", order_at_empty, "| value iteration says", learned)
if order_at_empty is None:
    print("NOT YET — fill order_at_empty")
else:
    print("CHECK 2 —", "PASS" if order_at_empty == learned else "LOOK — discuss why you differ")


## Stop 3 — Monte Carlo vs TD (one episode)

Often you do **not** have `P(d)`. You have days of experience.

- **Monte Carlo** waits until the episode ends, then assigns the return.  
- **TD** updates after one step using the next state's current value.


In [ ]:
# STOP 3 — RUN WITH ME · one episode: MC return + one TD target

def behavior_order(stock):
    return 1 if stock < 2 else 0


episode = []
stock = 1
# Fixed demand bins so the board matches on rerun
for demand in (2, 0, 1, 2):
    order = behavior_order(stock)
    ending, pay = transition(stock, order, demand)
    episode.append((stock, pay, ending))
    stock = ending

# Monte Carlo: walk backward G_t = r_t + gamma * G_{t+1}
monte_carlo = []
g = 0.0
for stock_t, pay, _ in reversed(episode):
    g = pay + GAMMA * g
    monte_carlo.append((stock_t, g))
monte_carlo.reverse()
print("Monte Carlo targets (stock, return):")
for stock_t, target in monte_carlo:
    print(f"  stock {stock_t}: {target:.2f}")

stock0, pay0, next_stock = episode[0][0], episode[0][1], episode[0][2]
td_target = pay0 + GAMMA * values[next_stock]
print(f"TD target for stock {stock0}: {td_target:.2f}  (current V={values[stock0]:.2f})")
print("Session 8 Q-learning = TD for state-action values — no P(d) in the update.")


## Stop 4 — When DP is the wrong tool (Lab 4 Monday bridge)

![Q-learning update preview](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

You cannot run the Bellman backup if:
- demand probabilities are **unknown**,
- the state includes every SKU, or
- competitors change the transition.

That is normal in operations. **Q-learning** (Session 8 / Lab 4) is model-free: it learns Q(s,a) from samples with ε-greedy exploration.

| Tonight | Next session |
| --- | --- |
| DP value table with known P(d) | Q-table trained on Electronics days — **no** P(d) |

Say to managers: DP is a whiteboard. Samples cost money because the agent tries bad orders.


## Stop 5 — Framework line (TRY THIS)

Hint: value iteration owns the **numeric** order (**RL**). Promo emails are still LLM + a person (Session 6 memo).


In [ ]:
# STOP 5 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL: one sentence you will keep
print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
if not framework_family or not framework_line:
    print("NOT YET")
else:
    print("CHECK 5 — recorded")


## Raise a hand — Session 7 complete

1. [ ] Can say what V(s) means in one sentence  
2. [ ] Value-iteration plot discussed  
3. [ ] `order_at_empty` matches (or you can defend a difference)  
4. [ ] Can contrast Monte Carlo vs TD  
5. [ ] Can say why Session 8 drops P(d)  
